### What this notebook does

1. Loads and combines the two annual sheets in the Excel workbook.
2. Profiles the raw data before making changes.
3. Cleans invalid, incomplete, cancelled, and duplicated transactions.
4. Creates financial, temporal, geographic, and order-level features.
5. Validates the cleaned transaction data.
6. Builds a customer-level table and assigns RFM scores and segments.
7. Saves the cleaned transactions, customer summary, and audit report.

## 0. Imports and project setup

Import the libraries used in the analysis and locate the project-level `Data/` directory. Unlike the original `.py` script, a notebook does not define `__file__`, so the project root is detected from the current working directory and its parent folders.

In [8]:
# Standard library
import warnings
from datetime import datetime
from pathlib import Path

# Third-party libraries
import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore")

# Find the nearest parent directory containing the project's Data folder.
CURRENT_DIR = Path.cwd().resolve()
ROOT = next(
    (folder for folder in [CURRENT_DIR, *CURRENT_DIR.parents] if (folder / "Data").is_dir()),
    None,
)

# If the Data folder does not exist yet, use the current working directory.
if ROOT is None:
    ROOT = CURRENT_DIR.parent if CURRENT_DIR.name.lower() == "notebooks" else CURRENT_DIR

DATA_DIR = ROOT / "Data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

RAW_FILE = DATA_DIR / "online_retail_II.xlsx"
CLEAN_OUT = DATA_DIR / "online_retail_clean.csv"
CUST_OUT = DATA_DIR / "customer_summary.csv"
REPORT_OUT = DATA_DIR / "data_quality_report.txt"

print(f"Project root : {ROOT}")
print(f"Data folder  : {DATA_DIR}")
print(f"Input file   : {RAW_FILE}")


Project root : C:\Users\FixTest\streaming_platform_customer_churn\retail-sales
Data folder  : C:\Users\FixTest\streaming_platform_customer_churn\retail-sales\Data
Input file   : C:\Users\FixTest\streaming_platform_customer_churn\retail-sales\Data\online_retail_II.xlsx


## 1. Audit log utility

`AuditLog` records the number of rows removed at each cleaning step, the percentage removed, and the number of rows remaining. The same log is printed during execution and later written to `data_quality_report.txt`.

In [9]:
class AuditLog:
    """Accumulates cleaning decisions and writes them to a text report."""

    def __init__(self):
        self._lines = []
        self._start = datetime.now()

    def log(self, step: str, rows_before: int, rows_after: int, note: str = ""):
        removed = rows_before - rows_after
        pct     = removed / rows_before * 100 if rows_before else 0
        line    = (
            f"  [{step:<35}]  "
            f"removed {removed:>7,} rows  ({pct:5.1f}%)  "
            f"→ {rows_after:>7,} remaining"
            + (f"  // {note}" if note else "")
        )
        self._lines.append(line)
        print(line)

    def write(self, path: Path) -> None:
        header = (
            "=" * 78 + "\n"
            " DATA QUALITY REPORT — Retail Sales Intelligence System\n"
            f" Generated: {self._start.strftime('%Y-%m-%d %H:%M:%S')}\n"
            "=" * 78 + "\n\n"
            "Cleaning Log\n"
            "────────────\n"
        )
        with open(path, "w") as f:
            f.write(header + "\n".join(self._lines) + "\n")
        print(f"\n  Audit log → {path}")

## 2. Load the raw workbook

The workbook contains two sheets: `Year 2009-2010` and `Year 2010-2011`. They are read separately and then concatenated into one DataFrame. `Customer ID` is loaded as text to avoid unwanted numeric conversion.

In [10]:
def load_raw(filepath: Path) -> pd.DataFrame:
    """
    Concatenate both annual sheets from the Online Retail II workbook.

    Customer ID is read as string to preserve any leading zeros that
    would be silently dropped by pandas' default int coercion.
    """
    print("\n── Step 1: Load ─────────────────────────────────────────")
    if not filepath.exists():
        raise FileNotFoundError(
            f"\n[ERROR] Raw file not found at {filepath}\n"
            "Download it from: https://archive.ics.uci.edu/dataset/502/online+retail+ii\n"
            "Place it in the Data/ folder and re-run.\n"
        )

    sheets = {}
    for sheet in ("Year 2009-2010", "Year 2010-2011"):
        sheets[sheet] = pd.read_excel(
            filepath, sheet_name=sheet, dtype={"Customer ID": str}
        )
        print(f"  {sheet}: {len(sheets[sheet]):>9,} rows")

    df = pd.concat(sheets.values(), ignore_index=True)
    print(f"  Combined total: {len(df):>9,} rows  ×  {df.shape[1]} columns")
    return df

In [11]:
# Load both annual sheets and combine them into one dataset.
df_raw = load_raw(RAW_FILE)

print("Raw data preview:")
display(df_raw.head())

print(f"\nDataset shape: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")


── Step 1: Load ─────────────────────────────────────────
  Year 2009-2010:   525,461 rows
  Year 2010-2011:   541,910 rows
  Combined total: 1,067,371 rows  ×  8 columns
Raw data preview:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom



Dataset shape: 1,067,371 rows × 8 columns


## 3. Raw data profile

Review the dataset before cleaning. This profile reports its size, memory usage, date range, unique entities, cancellations, missing values, invalid quantities or prices, and exact duplicate rows.

In [12]:
def profile_raw(df: pd.DataFrame) -> None:
    """Print a structured data quality snapshot of the raw frame."""
    print("\n── Step 2: Raw Data Profile ─────────────────────────────")
    print(f"  Shape              : {df.shape}")
    print(f"  Memory             : {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
    print(f"  Date range         : {df['InvoiceDate'].min()} → {df['InvoiceDate'].max()}")
    print(f"  Unique invoices    : {df['Invoice'].nunique():,}")
    print(f"  Unique customers   : {df['Customer ID'].nunique():,}")
    print(f"  Unique countries   : {df['Country'].nunique():,}")
    print(f"  Cancellations (C-) : {df['Invoice'].astype(str).str.startswith('C').sum():,}")
    print()

    print("  Missing values:")
    missing = df.isnull().sum()
    for col, cnt in missing[missing > 0].items():
        print(f"    {col:<20} : {cnt:>7,}  ({cnt/len(df)*100:.1f}%)")

    print()
    print("  Numeric anomalies:")
    print(f"    Negative Quantity  : {(df['Quantity'] < 0).sum():,}")
    print(f"    Zero/Neg Price     : {(df['Price'] <= 0).sum():,}")
    print(f"    Exact duplicates   : {df.duplicated().sum():,}")

In [13]:
profile_raw(df_raw)

# A compact missing-value table for easier inspection in the notebook.
missing_summary = (
    df_raw.isna().sum()
    .rename("missing_values")
    .to_frame()
    .query("missing_values > 0")
    .sort_values("missing_values", ascending=False)
)

display(missing_summary)


── Step 2: Raw Data Profile ─────────────────────────────
  Shape              : (1067371, 8)
  Memory             : 329.8 MB
  Date range         : 2009-12-01 07:45:00 → 2011-12-09 12:50:00
  Unique invoices    : 53,628
  Unique customers   : 5,942
  Unique countries   : 43
  Cancellations (C-) : 19,494

  Missing values:
    Description          :   4,382  (0.4%)
    Customer ID          : 243,007  (22.8%)

  Numeric anomalies:
    Negative Quantity  : 22,950
    Zero/Neg Price     : 6,207
    Exact duplicates   : 34,335


,missing_values
Customer ID,243007
Description,4382


## 4. Clean the transaction data

The cleaning function applies the original sequence of operations:

- Standardise column names to `snake_case`.
- Parse invoice dates.
- Remove cancellation invoices (invoice numbers beginning with `C`).
- Remove rows without a customer ID.
- Remove non-positive quantities and prices.
- Remove exact duplicate rows.
- Trim and standardise text fields.

Each row-removal step is recorded in the audit log with its business rationale.

In [26]:
def clean(df: pd.DataFrame, audit: AuditLog) -> pd.DataFrame:
    """
    Apply seven sequential cleaning steps, logging each to the audit trail.

    Steps
    ─────
    1. Standardise column names → snake_case
    2. Parse InvoiceDate to datetime
    3. Remove cancellations (Invoice prefix 'C')
    4. Drop rows missing Customer ID
    5. Remove invalid Quantity (≤ 0) and Price (≤ 0)
    6. Drop exact duplicate rows
    7. Normalise text fields (strip whitespace, title-case Description)

    Business justification is embedded in each log entry so the audit
    trail is self-documenting for any stakeholder review.
    """
    print("\n── Step 3: Cleaning ─────────────────────────────────────")

    # 3.1  Standardise column names
    df.columns = (
        df.columns
          .str.strip()
          .str.lower()
          .str.replace(" ", "_", regex=False)
          .str.replace(r"[^a-z0-9_]", "", regex=True)
    )

    # 3.2  Parse dates
    df["invoicedate"] = pd.to_datetime(df["invoicedate"], errors="coerce")
    n_bad_dates = df["invoicedate"].isna().sum()
    if n_bad_dates:
        before = len(df)
        df = df.dropna(subset=["invoicedate"])
        audit.log("Unparseable InvoiceDate", before, len(df), "coerce→NaT then drop")

    # 3.3  Cancellations
    before = len(df)
    mask_cancel = df["invoice"].astype(str).str.startswith("C")
    df = df[~mask_cancel].copy()
    audit.log("Cancellation invoices (C-*)", before, len(df),
              "Not revenue events; excluded from sales analysis")

    # 3.4  Missing Customer ID
    before = len(df)
    df = df.dropna(subset=["customer_id"])
    audit.log("Missing Customer ID", before, len(df),
              "No customer → no RFM segmentation or CLV calculation possible")

    # 3.5  Invalid Quantity
    before = len(df)
    df = df[df["quantity"] > 0].copy()
    audit.log("Non-positive Quantity", before, len(df),
              "Returns/adjustments; not forward sales")

    # 3.6  Invalid Price
    before = len(df)
    df = df[df["price"] > 0].copy()
    audit.log("Zero or negative Price", before, len(df),
              "Free samples / pricing errors; distort margin analysis")

    # 3.7  Exact duplicates
    before = len(df)
    df = df.drop_duplicates()
    audit.log("Exact duplicate rows", before, len(df),
              "System duplication artefacts")

    # 3.8  Normalise text
    df["customer_id"]  = df["customer_id"].astype(str).str.strip()
    df["stockcode"]    = df["stockcode"].astype(str).str.strip().str.upper()
    df["description"]  = df["description"].astype(str).str.strip().str.title()
    df["country"]      = df["country"].astype(str).str.strip()

    print(f"\n  ✓ Final clean rows: {len(df):,}")
    return df.reset_index(drop=True)

In [27]:
audit = AuditLog()
df_clean = clean(df_raw.copy(), audit)

print("Cleaned data preview:")
display(df_clean.head())

print(f"\nCleaned dataset shape: {df_clean.shape[0]:,} rows × {df_clean.shape[1]} columns")


── Step 3: Cleaning ─────────────────────────────────────
  [Cancellation invoices (C-*)        ]  removed  19,494 rows  (  1.8%)  → 1,047,877 remaining  // Not revenue events; excluded from sales analysis
  [Missing Customer ID                ]  removed 242,257 rows  ( 23.1%)  → 805,620 remaining  // No customer → no RFM segmentation or CLV calculation possible
  [Non-positive Quantity              ]  removed       0 rows  (  0.0%)  → 805,620 remaining  // Returns/adjustments; not forward sales
  [Zero or negative Price             ]  removed      71 rows  (  0.0%)  → 805,549 remaining  // Free samples / pricing errors; distort margin analysis
  [Exact duplicate rows               ]  removed  26,124 rows  (  3.2%)  → 779,425 remaining  // System duplication artefacts

  ✓ Final clean rows: 779,425
Cleaned data preview:


,invoice,stockcode,description,quantity,invoicedate,price,customer_id,country
0,489434,85048,15Cm Christmas Glass Ball 20 Lights,12,2009-12-01 07:45:00,6.95,13085,United Kingdom
1,489434,79323P,Pink Cherry Lights,12,2009-12-01 07:45:00,6.75,13085,United Kingdom
2,489434,79323W,White Cherry Lights,12,2009-12-01 07:45:00,6.75,13085,United Kingdom
3,489434,22041,"Record Frame 7"" Single Size",48,2009-12-01 07:45:00,2.10,13085,United Kingdom
4,489434,21232,Strawberry Ceramic Trinket Box,24,2009-12-01 07:45:00,1.25,13085,United Kingdom



Cleaned dataset shape: 779,425 rows × 8 columns


## 5. Feature engineering

Create the transaction-level fields used by downstream sales analysis:

| Feature group | Fields |
|---|---|
| Financial | `total_price`, `revenue` |
| Temporal | `year`, `month`, `month_name`, `quarter`, `week`, `day_of_week`, `day_name`, `is_weekend`, `hour` |
| Geographic | `country_clean` |
| Order-level | `order_id` |

`total_price` is calculated as `quantity × price`, and `revenue` is an alias of that value.

In [34]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Derive 15 analytics-ready columns from the cleaned transaction data.

    Financial
    ─────────
    total_price    : quantity × price  (line-level revenue; named to match
                     the original UCI column convention)
    revenue        : alias for total_price (used in all downstream queries)

    Temporal
    ────────
    year, month, quarter, week, day_of_week, hour
    month_name     : abbreviated month label ('Jan' … 'Dec')
    is_weekend     : True for Saturday and Sunday
    day_name       : full day name ('Monday' … 'Sunday')

    Geographic
    ──────────
    country_clean  : normalised country name (maps legacy codes)

    Order-level
    ───────────
    order_id       : string alias for invoice (clarity in GROUP BY)
    """
    print("\n── Step 4: Feature Engineering ─────────────────────────")

    # Financial
    df["total_price"] = (df["quantity"] * df["price"]).round(2)
    df["revenue"]     = df["total_price"].round(2)   # convenience alias

    # Temporal
    dt = df["invoicedate"].dt
    df["year"]        = dt.year
    df["month"]       = dt.month
    df["month_name"]  = dt.strftime("%b")
    df["quarter"]     = dt.quarter.map({1: "Q1", 2: "Q2", 3: "Q3", 4: "Q4"})
    df["week"]        = dt.isocalendar().week.astype(int)
    df["day_of_week"] = dt.dayofweek          # 0 = Monday … 6 = Sunday
    df["day_name"]    = dt.strftime("%A")
    df["is_weekend"]  = df["day_of_week"].isin([5, 6])
    df["hour"]        = dt.hour

    # Geographic
    country_map = {
        "EIRE":          "Ireland",
        "RSA":           "South Africa",
        "USA":           "United States",
        "Channel Islands": "Channel Islands",
    }
    df["country_clean"] = df["country"].replace(country_map)

    # Order alias
    df["order_id"] = df["invoice"].astype(str)

    new_cols = ["total_price","revenue","year","month","month_name","quarter",
                "week","day_of_week","day_name","is_weekend","hour",
                "country_clean","order_id"]
    print(f"  ✓ {len(new_cols)} new columns added: {new_cols}")
    return df

In [35]:
df_clean = engineer_features(df_clean)

new_feature_columns = [
    "total_price", "revenue", "year", "month", "month_name", "quarter",
    "week", "day_of_week", "day_name", "is_weekend", "hour",
    "country_clean", "order_id",
]

display(df_clean[["invoicedate", "quantity", "price", *new_feature_columns]].head())


── Step 4: Feature Engineering ─────────────────────────
  ✓ 13 new columns added: ['total_price', 'revenue', 'year', 'month', 'month_name', 'quarter', 'week', 'day_of_week', 'day_name', 'is_weekend', 'hour', 'country_clean', 'order_id']


,invoicedate,quantity,price,total_price,revenue,year,month,month_name,quarter,week,day_of_week,day_name,is_weekend,hour,country_clean,order_id
0,2009-12-01 07:45:00,12,6.95,83.4,83.4,2009,12,Dec,Q4,49,1,Tuesday,False,7,United Kingdom,489434
1,2009-12-01 07:45:00,12,6.75,81.0,81.0,2009,12,Dec,Q4,49,1,Tuesday,False,7,United Kingdom,489434
2,2009-12-01 07:45:00,12,6.75,81.0,81.0,2009,12,Dec,Q4,49,1,Tuesday,False,7,United Kingdom,489434
3,2009-12-01 07:45:00,48,2.10,100.8,100.8,2009,12,Dec,Q4,49,1,Tuesday,False,7,United Kingdom,489434
4,2009-12-01 07:45:00,24,1.25,30.0,30.0,2009,12,Dec,Q4,49,1,Tuesday,False,7,United Kingdom,489434


## 6. Validate the cleaned dataset

Run the original post-cleaning checks. The notebook raises an `AssertionError` if any invariant fails, preventing the customer summary from being built on data that did not pass validation.

In [45]:
def validate(df: pd.DataFrame) -> None:
    """
    Post-cleaning assertions.  Raises AssertionError immediately if
    any invariant is violated so downstream scripts never run on bad data.
    """
    print("\n── Step 5: Validation ───────────────────────────────────")
    checks = {
        "No missing customer_id"  : df["customer_id"].notna().all(),
        "No zero/neg revenue"     : (df["revenue"] > -1).all(),
        "No cancellation invoices": ~df["invoice"].astype(str).str.startswith("C").any(),
        "No negative quantity"    : (df["quantity"] > 0).all(),
        "InvoiceDate is datetime" : pd.api.types.is_datetime64_any_dtype(df["invoicedate"]),
        "revenue = qty × price"   : (
            (df["total_price"] - df["quantity"] * df["price"]).abs().max() < 0.01
        ),
    }
    all_pass = True
    for check, result in checks.items():
        status = "✓" if result else "✗ FAILED"
        print(f"  {status}  {check}")
        if not result:
            all_pass = False

    if not all_pass:
        raise AssertionError("Validation failed — review cleaning logic before proceeding.")
    print("\n  All validation checks passed.\n")

In [46]:
validate(df_clean)


── Step 5: Validation ───────────────────────────────────
  ✓  No missing customer_id
  ✓  No zero/neg revenue
  ✓  No cancellation invoices
  ✓  No negative quantity
  ✓  InvoiceDate is datetime
  ✓  revenue = qty × price

  All validation checks passed.



## 7. Build the customer summary and RFM segments

The transaction table is aggregated to one row per customer.

### RFM methodology

- **Recency:** days since the customer's last purchase. A lower value is better.
- **Frequency:** number of distinct invoices.
- **Monetary:** total lifetime revenue.

Each dimension receives a quintile score from 1 to 5. Recency is scored in reverse, so 5 represents the most recent customers. The three scores are added to form an `rfm_score` from 3 to 15.

| Segment | RFM score |
|---|---:|
| Champions | 13–15 |
| Loyal Customers | 10–12 |
| Potential Loyalists | 7–9 |
| At Risk | 5–6 |
| Lost | 3–4 |

In [52]:
def build_customer_summary(df: pd.DataFrame) -> pd.DataFrame:
    """
    Aggregate to one row per customer and attach RFM scores and segment labels.

    RFM Methodology
    ───────────────
    Snapshot date  = latest InvoiceDate + 1 day (avoids zero recency)
    Recency        = days since last purchase (lower = more recent = better)
    Frequency      = distinct invoice count
    Monetary       = total lifetime revenue

    Scoring        Quintile-based (1–5 per dimension).
                   Recency inverted: 5 = most recent.
                   Ties in Frequency / Monetary broken by rank(method='first').

    RFM Score      R + F + M   range 3–15

    Segments
    ────────
    Champions          : 13–15   (buy often, recently, high spend)
    Loyal Customers    : 10–12
    Potential Loyalists:  7–9
    At Risk            :  5–6    (previously valuable, now inactive)
    Lost               :  3–4    (low score on all dimensions)
    """
    print("── Step 6: Customer Summary & RFM Scoring ───────────────")
    snapshot = df["invoicedate"].max() + pd.Timedelta(days=1)

    cust = (
        df.groupby("customer_id")
          .agg(
              total_revenue    = ("revenue",     "sum"),
              total_orders     = ("order_id",    "nunique"),
              total_items      = ("quantity",    "sum"),
              total_price      = ("total_price", "sum"),
              first_purchase   = ("invoicedate", "min"),
              last_purchase    = ("invoicedate", "max"),
              country          = ("country_clean", "first"),
          )
          .reset_index()
    )

    cust["avg_order_value"]    = (cust["total_revenue"] / cust["total_orders"]).round(2)
    cust["avg_items_per_order"]= (cust["total_items"]   / cust["total_orders"]).round(1)
    cust["recency_days"]       = (snapshot - cust["last_purchase"]).dt.days
    cust["tenure_days"]        = (cust["last_purchase"] - cust["first_purchase"]).dt.days

    # RFM quintile scores
    cust["r_score"] = pd.qcut(
        cust["recency_days"], q=5, labels=[5, 4, 3, 2, 1], duplicates="drop"
    ).astype(int)
    cust["f_score"] = pd.qcut(
        cust["total_orders"].rank(method="first"), q=5, labels=[1, 2, 3, 4, 5]
    ).astype(int)
    cust["m_score"] = pd.qcut(
        cust["total_revenue"].rank(method="first"), q=5, labels=[1, 2, 3, 4, 5]
    ).astype(int)

    cust["rfm_score"] = cust["r_score"] + cust["f_score"] + cust["m_score"]

    def _segment(score: int) -> str:
        if score >= 13: return "Champions"
        if score >= 10: return "Loyal Customers"
        if score >= 7:  return "Potential Loyalists"
        if score >= 5:  return "At Risk"
        return "Lost"

    cust["segment"] = cust["rfm_score"].apply(_segment)

    # Summary
    print(f"  ✓ Customers built : {len(cust):,}")
    print(f"  Snapshot date     : {snapshot.date()}")
    print()
    print("  Segment distribution:")
    dist = cust["segment"].value_counts()
    for seg, cnt in dist.items():
        rev = cust.loc[cust["segment"] == seg, "total_revenue"].sum()
        print(f"    {seg:<22} : {cnt:>5,} customers  |  "
              f"£{rev:>10,.0f} lifetime revenue")

    return cust

In [53]:
df_cust = build_customer_summary(df_clean)

print("Customer summary preview:")
display(df_cust.head(10))

── Step 6: Customer Summary & RFM Scoring ───────────────
  ✓ Customers built : 5,878
  Snapshot date     : 2011-12-10

  Segment distribution:
    Potential Loyalists    : 1,453 customers  |  £ 1,238,840 lifetime revenue
    Loyal Customers        : 1,357 customers  |  £ 3,070,957 lifetime revenue
    Champions              : 1,295 customers  |  £12,533,230 lifetime revenue
    At Risk                :   967 customers  |  £   373,208 lifetime revenue
    Lost                   :   806 customers  |  £   158,570 lifetime revenue
Customer summary preview:


,customer_id,total_revenue,total_orders,total_items,total_price,first_purchase,last_purchase,country,avg_order_value,avg_items_per_order,recency_days,tenure_days,r_score,f_score,m_score,rfm_score,segment
0,12346,77556.46,12,74285,77556.46,2009-12-14 08:34:00,2011-01-18 10:01:00,United Kingdom,6463.04,6190.4,326,400,2,5,5,12,Loyal Customers
1,12347,4921.53,8,2967,4921.53,2010-10-31 14:20:00,2011-12-07 15:52:00,Iceland,615.19,370.9,2,402,5,4,5,14,Champions
2,12348,2019.40,5,2714,2019.40,2010-09-27 14:59:00,2011-09-25 13:13:00,Finland,403.88,542.8,75,362,3,4,4,11,Loyal Customers
3,12349,4428.69,4,1624,4428.69,2010-04-29 13:20:00,2011-11-21 09:51:00,Italy,1107.17,406.0,19,570,5,3,5,13,Champions
4,12350,334.40,1,197,334.40,2011-02-02 16:01:00,2011-02-02 16:01:00,Norway,334.40,197.0,310,0,2,1,2,5,At Risk
5,12351,300.93,1,261,300.93,2010-11-29 15:23:00,2010-11-29 15:23:00,Unspecified,300.93,261.0,375,0,2,1,2,5,At Risk
6,12352,2849.84,10,724,2849.84,2010-11-12 10:20:00,2011-11-03 14:37:00,Norway,284.98,72.4,36,356,4,5,4,13,Champions
7,12353,406.76,2,212,406.76,2010-10-27 12:44:00,2011-05-19 17:47:00,Bahrain,203.38,106.0,204,204,2,2,2,6,At Risk
8,12354,1079.40,1,530,1079.40,2011-04-21 13:11:00,2011-04-21 13:11:00,Spain,1079.40,530.0,232,0,2,1,3,6,At Risk
9,12355,947.61,2,543,947.61,2010-05-21 11:59:00,2011-05-09 13:49:00,Bahrain,473.80,271.5,214,353,2,2,3,7,Potential Loyalists


In [54]:
# Summarise customer count and lifetime revenue by RFM segment.
segment_summary = (
    df_cust.groupby("segment", as_index=False)
    .agg(
        customers=("customer_id", "count"),
        lifetime_revenue=("total_revenue", "sum"),
        average_recency_days=("recency_days", "mean"),
    )
    .sort_values("customers", ascending=False)
)

segment_summary["lifetime_revenue"] = segment_summary["lifetime_revenue"].round(2)
segment_summary["average_recency_days"] = segment_summary["average_recency_days"].round(1)
display(segment_summary)

,segment,customers,lifetime_revenue,average_recency_days
4,Potential Loyalists,1453,1238840.19,201.7
3,Loyal Customers,1357,3070956.80,95.7
1,Champions,1295,12533229.87,25.8
0,At Risk,967,373207.51,320.7
2,Lost,806,158569.88,517.2


## 8. Save outputs

Save the three deliverables to the project's `Data/` folder:

- `online_retail_clean.csv` — cleaned transaction-level data with engineered features.
- `customer_summary.csv` — customer-level aggregates, RFM scores, and segments.
- `data_quality_report.txt` — audit trail of the cleaning steps.

The CSV files are written without pandas' index column.

In [ ]:
def save_outputs(df_clean: pd.DataFrame, df_cust: pd.DataFrame) -> None:
    print("\n── Step 7: Save Outputs ─────────────────────────────────")
    df_clean.to_csv(CLEAN_OUT, index=False)
    df_cust.to_csv(CUST_OUT,   index=False)
    print(f"  ✓ {CLEAN_OUT.name:<35} {df_clean.shape[0]:>8,} rows  × {df_clean.shape[1]} cols"
          f"  ({CLEAN_OUT.stat().st_size / 1e6:.1f} MB)")
    print(f"  ✓ {CUST_OUT.name:<35} {df_cust.shape[0]:>8,} rows  × {df_cust.shape[1]} cols")

In [ ]:
save_outputs(df_clean, df_cust)
audit.write(REPORT_OUT)

print("\nOutput files:")
for output_path in [CLEAN_OUT, CUST_OUT, REPORT_OUT]:
    print(f"- {output_path.name}: {output_path.stat().st_size / 1_000_000:.2f} MB")